**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/peyrache-2015"], check=True)
    os.chdir("/content/encore/entries/peyrache-2015")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Peyrache et al. (2015), Fig. 5d,e: GLM prediction between ADn and PoS, from DANDI:000056

**Paper.** Peyrache A, Lacroix MM, Petersen PC, Buzsáki G (2015). Internally organized mechanisms of the head direction sense. *Nature Neuroscience* 18, 569-575. doi:10.1038/nn.3968.

**Dataset.** DANDI:000056, version 0.261106.1949, streamed from the archive (the version is discussed in `reproduce_peyrache_2015.ipynb`).

**Why a notebook of its own.** The GLM is one of the slowest analyses (the time of the whole run and of each session is printed below); it runs apart so that each notebook stays well under the 60 minute limit.

**Claims.** fig5d-adn-predicts-pos-reliably and fig5e-adn-predicts-pos are tested here. fig5e-readout-windows needs a grid of at least 20 readout windows per decade near 50 to 200 ms; it runs in `glm_windows_peyrache_2015.ipynb`.

**Method** (`hd.glm_session`; Methods, Information measures, and the criterion of fig5d-adn-predicts-pos-reliably). For each HD target cell of one structure, a Poisson GLM of its 10 ms spike counts on the counts of the other structure's HD cells convolved with a Gaussian readout window, with the modified exponential link (g-a, exp below 0 and linear above, ref 36), 10-fold cross-validated information in bits per spike against a homogeneous rate at the training mean. Readings run: (g-b) the assembly is the other structure's HD cells; (g-c) the targets are HD cells; the assembly-of-all-cells and all-cell-target readings are not run, for time. HD cells are those passing under reading (hd-a) or (hd-b); hd-a targets alone are shown as a variant. Readout windows (g-d): 7 windows from 10 to 1000 ms on a log scale. Deviations, for time: at most the first 20 minutes of each state are used, and the fit is by Newton's method (the log likelihood is concave, so its maximum is the one the Methods' derivative-free search finds).

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import time, resource
import numpy as np
import pandas as pd
from scipy import stats
import hd
pd.set_option("display.width", 200)
print("dandiset", hd.DANDISET, "version", hd.VERSION, "| readout windows (s):", np.round(hd.GLM_WINDOWS, 4).tolist(),
      "| at most", hd.GLM_MAX_S, "s of each state")

In [ ]:
# The GLM on every session of the mice whose files hold units labelled ADn and
# PoS (Mouse20 is among them; see the data check of the main notebook).
t0 = time.time()
G = hd.run_glm(["Mouse20", "Mouse24", "Mouse25", "Mouse28"], workers=4)
print(f"analysed {len(G)} sessions in {(time.time() - t0) / 60:.1f} min; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
GLM = pd.DataFrame([g for o in G for g in o["glm"]])
CELLS = pd.DataFrame([c for o in G for c in o["cells"]])

## Data check

In [ ]:
print("seconds per session:", {o["meta"]["path"].split("_ses-")[1].split("_")[0]: round(o["meta"]["seconds"]) for o in G})
hdc = CELLS[CELLS.hd_a | CELLS.hd_b]
print("HD cells (hd-a or hd-b) per session and structure, sessions with both structures:")
print(hdc.groupby(["session", "region"]).size().unstack(fill_value=0).to_string())
print("sessions with a GLM (HD cells in both structures):", GLM.session.nunique(), "by mouse:", GLM.groupby("subject").session.nunique().to_dict())
print("fits:", len(GLM), "| targets per state and structure:", GLM.groupby(["state", "target_region"]).target.size().div(len(hd.GLM_WINDOWS)).astype(int).to_dict())
print("targets left out for fewer than 20 spikes in a state:",
      int(sum(len(hdc[(hdc.session == s_) & (hdc.region == r_)]) for s_ in GLM.session.unique() for r_ in ("ADn", "PoS")) * 3
          - GLM.groupby(["session", "state", "target_region", "target"]).ngroups))
print("information range (bits/spike):", round(GLM["info"].min(), 3), "to", round(GLM["info"].max(), 3), "| non-finite:", int((~np.isfinite(GLM["info"])).sum()))

## fig5d-adn-predicts-pos-reliably

> ADn assemblies predict PoS spike trains reliably in all brain states.

Best window per target and state; targets kept when positively predicted at one window or more (criterion). Two-sided Wilcoxon signed-rank test of the best-window information against 0, per state; all targets, without the selection, as a variant.

In [ ]:
best = GLM.loc[GLM.groupby(["session", "state", "target_region", "target"])["info"].idxmax()]
best = best.assign(kept=best["info"] > 0)
rows = []
for tg in ("hd-a or hd-b", "hd-a"):
    b0 = best if tg == "hd-a or hd-b" else best[best.hd_a.astype(bool)]
    for st in ("wake", "SWS", "REM"):
        for sel in ("kept", "all targets"):
            b = b0[(b0.state == st) & (b0.target_region == "PoS")]
            if sel == "kept":
                b = b[b.kept]
            w = stats.wilcoxon(b["info"])
            rows.append(dict(targets=tg, state=st, selection=sel, n=len(b), median_info=b["info"].median(),
                             frac_positive=float((b["info"] > 0).mean()), wilcoxon_p=w.pvalue))
F5D = pd.DataFrame(rows)
print(F5D.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("expected 86 PoS cells; Wilcoxon p < 1e-10 in every state")
print("by mouse, kept PoS targets and median best-window information (hd-a or hd-b):")
print(best[best.kept & (best.target_region == "PoS")].groupby(["subject", "state"])["info"].agg(["size", "median"]).round(3).unstack().to_string())
print("median best window (s) over kept PoS targets, 7-window grid, not deciding fig5e-readout-windows:",
      best[best.kept & (best.target_region == "PoS")].groupby("state").window.median().round(3).to_dict())

**Result: reproduced.** ADn HD cell assemblies predict PoS HD cells above chance in every state. Of the PoS targets with HD cells under either reading, 122 (waking), 99 (SWS) and 101 (REM) are kept as positively predicted, with median best-window information of 0.67, 0.63 and 0.49 bits per spike and Wilcoxon p of 9.2e-22, 5.7e-18 and 2.7e-18, each below 1e-10. With hd-a targets alone, 81, 66 and 69 are kept (waking within the expected 86 plus or minus 8.6), with p of 5.4e-15, 1.6e-12 and 5.2e-13. Without the selection of positively predicted targets, which makes the test one-sided by construction, p is 5.0e-22, 2.5e-18 and 1.1e-18 (either reading), so the selection does not decide the result.

## fig5e-adn-predicts-pos

> PoS assemblies predict ADn spike trains more weakly than ADn assemblies predict PoS spike trains, in all brain states and especially during SWS.

Best-window information of PoS targets (ADn assemblies) against ADn targets (PoS assemblies), two-sided Mann-Whitney U-test per state; the gap is the difference of medians relative to the PoS median.

In [ ]:
rows = []
for tg in ("hd-a or hd-b", "hd-a"):
    b0 = best if tg == "hd-a or hd-b" else best[best.hd_a.astype(bool)]
    for sel in ("kept", "all targets"):
        for st in ("wake", "SWS", "REM"):
            b = b0[b0.state == st]
            if sel == "kept":
                b = b[b.kept]
            P, A = b[b.target_region == "PoS"]["info"], b[b.target_region == "ADn"]["info"]
            u = stats.mannwhitneyu(P, A, alternative="two-sided")
            rows.append(dict(targets=tg, selection=sel, state=st, n_pos=len(P), n_adn=len(A), median_pos=P.median(),
                             median_adn=A.median(), gap=(P.median() - A.median()) / P.median(), p=u.pvalue,
                             pos_higher=P.median() > A.median()))
F5E = pd.DataFrame(rows)
print(F5E.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for (tg, sel), g in F5E.groupby(["targets", "selection"]):
    print(f"{tg}, {sel}: PoS higher with p < 0.05 in {int(((g.p < 0.05) & g.pos_higher).sum())} of 3 states; largest gap in "
          f"{g.loc[g.gap.idxmax(), 'state']}")
print("expected 62 ADn cells; paper p: wake 3e-5, SWS < 1e-10, REM 1.1e-6")

**Result: reproduced.** With the kept targets of fig5d (HD cells under either reading), PoS targets are better predicted from ADn assemblies than ADn targets from PoS assemblies in every state: medians 0.67 against 0.55 bits per spike in waking (Mann-Whitney p = 0.0095), 0.63 against 0.17 in SWS (p = 6.3e-11) and 0.49 against 0.33 in REM (p = 0.020). The gap is largest in SWS (0.72 of the PoS median, against 0.19 in waking and 0.32 in REM). With hd-a targets the tests give p = 2.6e-4, 4.2e-9 and 0.014, the gap again largest in SWS. Without the selection (either-reading targets) the REM test gives p = 0.059, so that variant holds in two states. The ADn target counts (176 in waking, 162 with hd-a targets) are far above the expected 62, as the ADn HD cells of every session of the dual mice are targets here.

## fig5e-readout-windows

The criterion needs at least 20 readout windows per decade near 50 to 200 ms. It runs in `glm_windows_peyrache_2015.ipynb` with 26 windows, PoS targets only. The median best windows on the 7-window grid are printed above for information and decide nothing.

## Summary

| claim | outcome |
|---|---|
| fig5d-adn-predicts-pos-reliably | reproduced |
| fig5e-adn-predicts-pos | reproduced |

**How the analysis differs from the paper's.** Readings (g-b) and (g-c) with HD cells only; 7 readout windows; at most 20 minutes of each state; Newton's method in place of a derivative-free search.
